# FTD Physics Engine Test Suite

This notebook contains comprehensive tests for all 12 phases of the FTD simulation engine.

## Test Categories
1. **Time Gating** (Phase 1) - Relativistic time dilation
2. **Forces** (Phase 6) - All 5 force types and differential operators
3. **Integration** (Phase 7) - Velocity and position updates
4. **Movement** (Phase 8) - Discrete lattice movement
5. **Transmutation** (Phase 10) - Weak-force polarity flips
6. **Full Cycle** - All 12 phases together
7. **Conservation Laws** - Charge and vacuum stability
8. **Performance** - Timing benchmarks
9. **Differential Operators** - Mathematical validation

In [1]:
# Imports
import numpy as np
import sys
import time

# Add parent directory to path if needed
sys.path.insert(0, '../..')

from ternary_matrix.model.grid import Universe
from ternary_matrix.config import CONSTANTS, get_test_config
from ternary_matrix.physics import (
    tick,
    tick_minimal,
    run_simulation,
    get_diagnostics,
    # Time gating
    time_gate,
    get_effective_time_rate,
    # Forces
    calculate_density,
    gradient_3d,
    divergence_3d,
    curl_3d,
    smooth_field,
    gravity_force,
    coulomb_force,
    lorentz_force,
    accumulate_forces,
    weak_stress,
    # Integration
    integrate,
    clamp_velocity,
    get_max_speed,
    # Movement
    move_particles,
    # Waves
    propagate_flux,
    # Interactions
    process_interactions,
    get_annihilation_count,
    # Transmutation
    transmute,
    get_stress_field,
    # Binding
    update_bindings,
    count_neighbors_moore,
    get_triad_count,
)

print("All imports successful!")
print(f"Grid size: {CONSTANTS.GRID_SIZE}")
print(f"Speed of light (c): {CONSTANTS.C}")
print(f"Manifestation threshold (KB): {CONSTANTS.KB}")

All imports successful!
Grid size: 256
Speed of light (c): 0.5
Manifestation threshold (KB): 1.2


In [2]:
# Helper function to report test results
def run_test(name, test_func):
    """Run a test and report pass/fail."""
    try:
        test_func()
        print(f"✅ PASSED: {name}")
        return True
    except AssertionError as e:
        print(f"❌ FAILED: {name}")
        print(f"   Error: {e}")
        return False
    except Exception as e:
        print(f"💥 ERROR: {name}")
        print(f"   Exception: {type(e).__name__}: {e}")
        return False

# Track results
results = {'passed': 0, 'failed': 0, 'error': 0}

---
## Phase 1: Time Gating Tests

Time gating implements relativistic time dilation. Fast-moving voxels accumulate phase slower, so they update less frequently than stationary voxels.

In [3]:
def test_stationary_voxels_always_active():
    """Stationary voxels should always be active."""
    universe = Universe(size=16)
    universe.velocity.fill(0)
    
    time_gate(universe)
    
    assert np.all(universe.is_active), "All stationary voxels should be active"

run_test("Stationary voxels always active", test_stationary_voxels_always_active)

✅ PASSED: Stationary voxels always active


True

In [4]:
def test_fast_voxels_less_active():
    """Fast-moving voxels should update less frequently due to time dilation."""
    universe = Universe(size=16)
    
    # Set one voxel to high velocity (near c=0.5)
    center = universe.size // 2
    universe.velocity[center, center, center] = [0.4, 0, 0]
    
    # Run multiple time gates and track activity
    active_counts = []
    for _ in range(100):
        universe.phase_accum.fill(0)
        time_gate(universe)
        active_counts.append(universe.is_active[center, center, center])
    
    fast_active_rate = np.mean(active_counts)
    print(f"   Fast voxel active rate: {fast_active_rate:.2%} (should be < 100%)")
    assert fast_active_rate < 1.0, "Fast voxel should skip some ticks"

run_test("Fast voxels less active (time dilation)", test_fast_voxels_less_active)

   Fast voxel active rate: 0.00% (should be < 100%)
✅ PASSED: Fast voxels less active (time dilation)


True

In [5]:
def test_effective_time_rate():
    """Test the diagnostic time rate function."""
    universe = Universe(size=16)
    universe.velocity[5, 5, 5] = [0.3, 0, 0]
    
    rate = get_effective_time_rate(universe)
    
    stationary_rate = rate[0, 0, 0]
    moving_rate = rate[5, 5, 5]
    
    print(f"   Stationary voxel rate: {stationary_rate:.4f} (should be ~1.0)")
    print(f"   Moving voxel rate: {moving_rate:.4f} (should be < 1.0)")
    
    assert np.isclose(stationary_rate, 1.0), "Stationary should have rate ~1"
    assert moving_rate < 1.0, "Moving voxel should have rate < 1"

run_test("Effective time rate diagnostic", test_effective_time_rate)

   Stationary voxel rate: 1.0000 (should be ~1.0)
   Moving voxel rate: 0.8000 (should be < 1.0)
✅ PASSED: Effective time rate diagnostic


True

---
## Phase 6: Force Tests

Testing the 5 force types and discrete differential operators (gradient, divergence, curl).

In [6]:
def test_gradient_zero_for_uniform_field():
    """Gradient of uniform field should be zero."""
    uniform_field = np.ones((16, 16, 16), dtype=np.float32) * 5.0
    grad = gradient_3d(uniform_field)
    
    max_grad = np.max(np.abs(grad))
    print(f"   Max gradient magnitude: {max_grad:.2e} (should be ~0)")
    assert np.allclose(grad, 0, atol=1e-6), "Gradient of constant should be zero"

run_test("Gradient of uniform field is zero", test_gradient_zero_for_uniform_field)

   Max gradient magnitude: 0.00e+00 (should be ~0)
✅ PASSED: Gradient of uniform field is zero


True

In [7]:
def test_divergence_zero_for_constant_vector_field():
    """Divergence of constant vector field should be zero."""
    universe = Universe(size=16)
    universe.flux.fill(0)
    universe.flux[..., 0] = 1.0  # Constant x-component
    
    div = divergence_3d(universe.flux)
    
    max_div = np.max(np.abs(div))
    print(f"   Max divergence: {max_div:.2e} (should be ~0)")
    assert np.allclose(div, 0, atol=1e-6), "Divergence of constant field should be zero"

run_test("Divergence of constant vector field is zero", test_divergence_zero_for_constant_vector_field)

   Max divergence: 0.00e+00 (should be ~0)
✅ PASSED: Divergence of constant vector field is zero


True

In [8]:
def test_curl_of_gradient_is_zero():
    """Curl of a gradient should be zero (vector calculus identity: ∇×∇f = 0)."""
    # Create a scalar field f = x²
    x = np.arange(16, dtype=np.float32)
    scalar = x[:, None, None] ** 2
    scalar = np.broadcast_to(scalar, (16, 16, 16)).copy()
    
    # Compute gradient
    grad = gradient_3d(scalar)
    
    # Compute curl of gradient
    curl = curl_3d(grad)
    
    max_curl = np.max(np.abs(curl))
    print(f"   Max |∇×∇f|: {max_curl:.2e} (should be ~0)")
    assert np.allclose(curl, 0, atol=1e-5), "Curl of gradient should be zero"

run_test("Curl of gradient is zero (∇×∇f = 0)", test_curl_of_gradient_is_zero)

   Max |∇×∇f|: 0.00e+00 (should be ~0)
✅ PASSED: Curl of gradient is zero (∇×∇f = 0)


True

In [9]:
def test_gravity_attracts_to_density():
    """Gravity force should point toward high density regions."""
    universe = Universe(size=16)
    universe.flux.fill(0)
    universe.density.fill(0)
    
    # Create a density distribution (not just single point) so smoothing produces gradient
    center = universe.size // 2
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            for dz in range(-2, 3):
                x = (center + dx) % universe.size
                y = (center + dy) % universe.size
                z = (center + dz) % universe.size
                dist = abs(dx) + abs(dy) + abs(dz)
                universe.density[x, y, z] = max(0, 10.0 - dist * 2)
    
    f_grav = gravity_force(universe)
    
    nonzero_count = np.count_nonzero(f_grav)
    print(f"   Non-zero force components: {nonzero_count}")
    assert np.any(f_grav != 0), "Gravity force should be non-zero near density gradient"

run_test("Gravity attracts toward density", test_gravity_attracts_to_density)

   Non-zero force components: 720
✅ PASSED: Gravity attracts toward density


True

In [10]:
def test_coulomb_like_charges_repel():
    """Same-sign charges should repel each other."""
    universe = Universe(size=16)
    universe.charge.fill(0)
    universe.states.fill(0)
    
    # Create a charge distribution
    for x in range(4, 8):
        for y in range(7, 10):
            for z in range(7, 10):
                universe.charge[x, y, z] = 1.0
                universe.states[x, y, z] = 1
    
    f_coulomb = coulomb_force(universe)
    
    nonzero_count = np.count_nonzero(f_coulomb)
    print(f"   Non-zero Coulomb force components: {nonzero_count}")
    assert np.any(f_coulomb != 0), "Coulomb force should be non-zero near charge gradient"

run_test("Coulomb force from charge gradient", test_coulomb_like_charges_repel)

   Non-zero Coulomb force components: 84
✅ PASSED: Coulomb force from charge gradient


True

In [11]:
def test_force_accumulator_clears():
    """Force accumulator should be cleared after integration."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.charge[8, 8, 8] = 1.0
    universe.force_accum[8, 8, 8] = [1.0, 2.0, 3.0]
    universe.is_active.fill(True)
    
    integrate(universe)
    
    max_force = np.max(np.abs(universe.force_accum))
    print(f"   Max force after integration: {max_force:.2e} (should be 0)")
    assert np.allclose(universe.force_accum, 0), "Force accumulator should be cleared"

run_test("Force accumulator clears after integration", test_force_accumulator_clears)

   Max force after integration: 0.00e+00 (should be 0)
✅ PASSED: Force accumulator clears after integration


True

---
## Phase 7: Integration Tests

Testing velocity updates from forces and position remainder accumulation.

In [12]:
def test_velocity_from_force():
    """Velocity should increase from applied force."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.force_accum[8, 8, 8] = [1.0, 0, 0]
    universe.is_active.fill(True)
    
    initial_vel = universe.velocity[8, 8, 8, 0]
    integrate(universe)
    final_vel = universe.velocity[8, 8, 8, 0]
    
    print(f"   Initial velocity: {initial_vel}")
    print(f"   Final velocity: {final_vel}")
    assert final_vel > initial_vel, "Velocity should increase from force"

run_test("Velocity increases from force", test_velocity_from_force)

   Initial velocity: 0.0
   Final velocity: 0.5
✅ PASSED: Velocity increases from force


True

In [13]:
def test_speed_limit_enforced():
    """Velocity should be clamped to speed of light."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.velocity[8, 8, 8] = [1.0, 1.0, 1.0]  # Way above c=0.5
    universe.is_active.fill(True)
    
    clamp_velocity(universe)
    
    max_speed = get_max_speed(universe)
    print(f"   Max speed after clamping: {max_speed:.4f} (c = {CONSTANTS.C})")
    assert max_speed <= CONSTANTS.C + 1e-6, f"Speed should not exceed c={CONSTANTS.C}"

run_test("Speed limit enforced (v ≤ c)", test_speed_limit_enforced)

   Max speed after clamping: 0.5000 (c = 0.5)
✅ PASSED: Speed limit enforced (v ≤ c)


True

In [14]:
def test_position_remainder_accumulates():
    """Position remainder should accumulate from velocity."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.velocity[8, 8, 8] = [0.1, 0, 0]
    universe.is_active.fill(True)
    
    initial_rem = universe.position_rem[8, 8, 8, 0]
    integrate(universe)
    final_rem = universe.position_rem[8, 8, 8, 0]
    
    print(f"   Initial remainder: {initial_rem}")
    print(f"   Final remainder: {final_rem}")
    assert final_rem > initial_rem, "Position remainder should accumulate"

run_test("Position remainder accumulates", test_position_remainder_accumulates)

   Initial remainder: 0.0
   Final remainder: 0.10000000149011612
✅ PASSED: Position remainder accumulates


True

---
## Phase 8: Movement Tests

Testing discrete lattice movement when position remainder exceeds threshold.

In [15]:
def test_movement_when_remainder_exceeds_one():
    """Particle should move when position remainder >= 1."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.position_rem[8, 8, 8] = [1.5, 0, 0]  # Should move +x
    
    move_particles(universe)
    
    old_state = universe.states[8, 8, 8]
    new_state = universe.states[9, 8, 8]
    
    print(f"   Old position state: {old_state} (should be 0)")
    print(f"   New position state: {new_state} (should be 1)")
    assert old_state == 0, "Original position should be void"
    assert new_state == 1, "New position should have particle"

run_test("Movement when remainder >= 1", test_movement_when_remainder_exceeds_one)

   Old position state: 0 (should be 0)
   New position state: 1 (should be 1)
✅ PASSED: Movement when remainder >= 1


True

In [16]:
def test_toroidal_boundary():
    """Movement should wrap around boundaries (toroidal)."""
    universe = Universe(size=16)
    edge = universe.size - 1  # x = 15
    universe.states[edge, 8, 8] = 1
    universe.position_rem[edge, 8, 8] = [1.5, 0, 0]  # Should wrap to x=0
    
    move_particles(universe)
    
    wrapped_state = universe.states[0, 8, 8]
    print(f"   State at x=0 after wrap: {wrapped_state} (should be 1)")
    assert wrapped_state == 1, "Particle should wrap to x=0"

run_test("Toroidal boundary wrapping", test_toroidal_boundary)

   State at x=0 after wrap: 1 (should be 1)
✅ PASSED: Toroidal boundary wrapping


True

In [17]:
def test_no_movement_to_occupied():
    """Particle should not move into occupied same-sign voxel directly."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.states[9, 8, 8] = 1  # Occupied by same sign
    universe.position_rem[8, 8, 8] = [1.5, 0, 0]
    
    move_particles(universe)
    
    # Both should still exist (elastic collision)
    total = (universe.states[8, 8, 8] != 0) + (universe.states[9, 8, 8] != 0)
    print(f"   Particles remaining: {total} (should be >= 1)")
    assert total >= 1, "At least one particle should remain"

run_test("No direct movement to occupied", test_no_movement_to_occupied)

   Particles remaining: True (should be >= 1)
✅ PASSED: No direct movement to occupied


True

---
## Phase 10: Transmutation Tests

Testing weak-force polarity flips under high field stress.

In [18]:
def test_no_transmutation_at_low_stress():
    """No transmutation should occur below stress threshold."""
    universe = Universe(size=16)
    universe.states[8, 8, 8] = 1
    universe.flux[8, 8, 8] = [0.1, 0, 0]  # Low flux = low stress
    
    calculate_density(universe)
    initial_state = universe.states[8, 8, 8]
    
    # Run many times
    for _ in range(100):
        transmute(universe)
    
    final_state = universe.states[8, 8, 8]
    print(f"   Initial state: {initial_state}")
    print(f"   Final state after 100 transmute calls: {final_state}")
    assert final_state == initial_state, "State should not flip at low stress"

run_test("No transmutation at low stress", test_no_transmutation_at_low_stress)

   Initial state: 1
   Final state after 100 transmute calls: 1
✅ PASSED: No transmutation at low stress


True

In [19]:
def test_stress_field_calculation():
    """Stress field should be computed correctly."""
    universe = Universe(size=16)
    
    # Create a flux gradient
    for i in range(universe.size):
        universe.flux[i, :, :, 0] = i * 0.5
    
    calculate_density(universe)
    stress = get_stress_field(universe)
    
    max_stress = np.max(stress)
    nonzero = np.count_nonzero(stress > 0)
    print(f"   Max stress: {max_stress:.4f}")
    print(f"   Voxels with stress > 0: {nonzero}")
    assert np.any(stress > 0), "Stress should be non-zero where gradients exist"

run_test("Stress field calculation", test_stress_field_calculation)

   Max stress: 7.0000
   Voxels with stress > 0: 4096
✅ PASSED: Stress field calculation


True

---
## Full Cycle Tests

Testing all 12 phases together.

In [20]:
def test_single_tick_runs():
    """A single tick should complete without error."""
    universe = Universe(size=16)
    universe.flux[8, 8, 8] = [2.0, 0, 0]
    calculate_density(universe)
    
    tick_count = tick(universe)
    
    print(f"   Tick count: {tick_count}")
    print(f"   Universe.tick: {universe.tick}")
    assert tick_count == 1, "Tick count should be 1"
    assert universe.tick == 1, "Universe tick should be 1"

run_test("Single tick completes", test_single_tick_runs)

   Tick count: 1
   Universe.tick: 1
✅ PASSED: Single tick completes


True

In [21]:
def test_1000_ticks_stability():
    """Simulation should remain stable over 1000 ticks."""
    universe = Universe(size=16)
    universe.flux[8, 8, 8] = [2.0, 2.0, 2.0]
    
    for i in range(1000):
        tick(universe)
        if i % 200 == 0:
            print(f"   Tick {i}: flux_max={np.max(np.abs(universe.flux)):.4f}")
    
    has_nan = np.any(np.isnan(universe.flux)) or np.any(np.isnan(universe.density))
    has_inf = np.any(np.isinf(universe.flux)) or np.any(np.isinf(universe.density))
    
    print(f"   Has NaN: {has_nan}")
    print(f"   Has Inf: {has_inf}")
    assert not has_nan, "No NaN values should appear"
    assert not has_inf, "No Inf values should appear"

run_test("1000 ticks stability", test_1000_ticks_stability)

   Tick 0: flux_max=0.9490
   Tick 200: flux_max=0.0000
   Tick 400: flux_max=0.0000
   Tick 600: flux_max=0.0000
   Tick 800: flux_max=0.0000
   Has NaN: False
   Has Inf: False
✅ PASSED: 1000 ticks stability


True

In [22]:
def test_diagnostics_function():
    """Diagnostics should return valid values."""
    universe = Universe(size=16)
    
    universe.states[5, 5, 5] = 1
    universe.states[10, 10, 10] = -1
    universe.charge[5, 5, 5] = 1.0
    universe.charge[10, 10, 10] = -1.0
    universe.flux[8, 8, 8] = [1.0, 1.0, 1.0]
    calculate_density(universe)
    
    diag = get_diagnostics(universe)
    
    print(f"   Diagnostics: {diag}")
    assert diag['tick'] == 0, "Tick should be 0"
    assert diag['manifested_count'] == 2, "Should have 2 manifested"
    assert diag['positive_count'] == 1, "Should have 1 positive"
    assert diag['negative_count'] == 1, "Should have 1 negative"
    assert np.isclose(diag['total_charge'], 0.0), "Net charge should be 0"

run_test("Diagnostics function", test_diagnostics_function)

   Diagnostics: {'tick': 0, 'manifested_count': np.int64(2), 'positive_count': np.int64(1), 'negative_count': np.int64(1), 'total_flux': np.float32(1.7320508), 'total_charge': np.float32(0.0), 'avg_speed': np.float32(0.0), 'max_speed': np.float32(0.0), 'kinetic_energy': np.float32(0.0)}
✅ PASSED: Diagnostics function


True

---
## Conservation Law Tests

Testing that physical conservation laws are maintained.

In [23]:
def test_charge_conservation():
    """Total charge should be conserved."""
    universe = Universe(size=32)
    
    # Create equal positive and negative particles
    universe.states[5, 5, 5] = 1
    universe.states[20, 20, 20] = -1
    universe.charge[5, 5, 5] = 1.0
    universe.charge[20, 20, 20] = -1.0
    universe.flux[5, 5, 5] = [2.0, 0, 0]
    universe.flux[20, 20, 20] = [2.0, 0, 0]
    
    initial_charge = universe.get_total_charge()
    print(f"   Initial charge: {initial_charge}")
    
    for _ in range(100):
        tick(universe)
    
    final_charge = universe.get_total_charge()
    print(f"   Final charge: {final_charge}")
    
    # Charge conserved or particles annihilated (both -> 0)
    assert np.isclose(initial_charge, final_charge, atol=1e-6) or \
           np.isclose(final_charge, 0, atol=1e-6), "Charge should be conserved"

run_test("Charge conservation", test_charge_conservation)

   Initial charge: 0.0
   Final charge: 0.0
✅ PASSED: Charge conservation


True

In [24]:
def test_vacuum_stability():
    """Empty lattice should remain empty."""
    universe = Universe(size=16)
    universe.reset()
    
    for _ in range(100):
        tick(universe)
    
    manifested = universe.get_manifested_count()
    print(f"   Manifested particles after 100 ticks: {manifested}")
    assert manifested == 0, "No spontaneous manifestation should occur"

run_test("Vacuum stability", test_vacuum_stability)

   Manifested particles after 100 ticks: 0
✅ PASSED: Vacuum stability


True

---
## Performance Tests

Timing benchmarks for the simulation.

In [25]:
def test_tick_completes_in_reasonable_time():
    """A single tick should complete quickly for 32³ grid."""
    universe = Universe(size=32)
    universe.flux[16, 16, 16] = [5.0, 5.0, 5.0]
    
    start = time.time()
    for _ in range(10):
        tick(universe)
    elapsed = time.time() - start
    
    ticks_per_sec = 10 / elapsed
    print(f"   10 ticks in {elapsed:.3f}s ({ticks_per_sec:.1f} ticks/sec)")
    assert elapsed < 5.0, f"Too slow: {elapsed:.2f}s for 10 ticks"

run_test("Performance: 10 ticks < 5s", test_tick_completes_in_reasonable_time)

   10 ticks in 0.058s (172.8 ticks/sec)
✅ PASSED: Performance: 10 ticks < 5s


True

In [26]:
def test_minimal_tick_faster_than_full():
    """Minimal tick should be faster than full tick."""
    universe = Universe(size=32)
    
    # Time full tick
    universe.reset()
    universe.flux[16, 16, 16] = [5.0, 5.0, 5.0]
    start = time.time()
    for _ in range(10):
        tick(universe)
    full_time = time.time() - start
    
    # Time minimal tick
    universe.reset()
    universe.flux[16, 16, 16] = [5.0, 5.0, 5.0]
    start = time.time()
    for _ in range(10):
        tick_minimal(universe)
    minimal_time = time.time() - start
    
    print(f"   Full tick: {full_time:.3f}s")
    print(f"   Minimal tick: {minimal_time:.3f}s")
    print(f"   Speedup: {full_time/minimal_time:.2f}x")
    assert minimal_time <= full_time * 1.5, "Minimal should not be much slower"

run_test("Minimal tick vs full tick", test_minimal_tick_faster_than_full)

   Full tick: 0.056s
   Minimal tick: 0.022s
   Speedup: 2.53x
✅ PASSED: Minimal tick vs full tick


True

---
## Differential Operators Tests

Mathematical validation of discrete operators.

In [27]:
def test_laplacian_of_constant_is_zero():
    """Laplacian of constant field should be zero."""
    from ternary_matrix.physics.waves import laplacian_3d_vector
    
    field = np.ones((16, 16, 16, 3), dtype=np.float32) * 5.0
    lap = laplacian_3d_vector(field)
    
    max_lap = np.max(np.abs(lap))
    print(f"   Max |∇²f|: {max_lap:.2e} (should be ~0)")
    assert np.allclose(lap, 0, atol=1e-6), "Laplacian of constant should be zero"

run_test("Laplacian of constant is zero", test_laplacian_of_constant_is_zero)

   Max |∇²f|: 0.00e+00 (should be ~0)
✅ PASSED: Laplacian of constant is zero


True

In [28]:
def test_smooth_field_reduces_variance():
    """Smoothing should reduce spatial variance."""
    np.random.seed(42)
    field = np.random.randn(16, 16, 16).astype(np.float32)
    initial_var = np.var(field)
    
    smoothed = smooth_field(field)
    final_var = np.var(smoothed)
    
    print(f"   Initial variance: {initial_var:.4f}")
    print(f"   Smoothed variance: {final_var:.4f}")
    print(f"   Reduction: {100*(1-final_var/initial_var):.1f}%")
    assert final_var < initial_var, "Smoothing should reduce variance"

run_test("Smoothing reduces variance", test_smooth_field_reduces_variance)

   Initial variance: 0.9939
   Smoothed variance: 0.1675
   Reduction: 83.1%
✅ PASSED: Smoothing reduces variance


True

---
## Summary

In [29]:
print("\n" + "="*60)
print("TEST SUITE COMPLETE")
print("="*60)
print("\nAll 26 tests should show ✅ PASSED above.")
print("\nIf any tests failed, review the error messages and fix the issues.")


TEST SUITE COMPLETE

All 26 tests should show ✅ PASSED above.

If any tests failed, review the error messages and fix the issues.
